# PRISM current-pipeline orientation comparison

This notebook runs the current `prism.py` pipeline with one frozen input CSV, one frozen template panel, and one frozen threshold configuration. It creates three explicitly comparable arms:

- `native_default`: omits `--orientation`; the CLI default is native dual-assignment behavior (`o1` + `o2`).
- `o1`: passes `--orientation o1`.
- `o2`: passes `--orientation o2`.

All arms use `--no-refine` so this first comparison localizes input, alignment, transformation, and clash differences. Set `RUN_PIPELINES = True` only after reviewing the generated commands. No arm runs by default.

In [1]:
from __future__ import annotations

import csv
import json
import hashlib
import itertools
import os
import shlex
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

REPO_ROOT = Path('/scratch/rshadi25/GitHub/PRISM-prescript')
CURRENT_CLI = REPO_ROOT / 'prism.py'
PYTHON = Path(sys.executable)
assert 'gtalign_env' in str(PYTHON), f'Use gtalign_env; got {PYTHON}'
assert CURRENT_CLI.exists()
sys.path.insert(0, str(REPO_ROOT))

from src.pipeline_inputs import select_templates
from src.stepwise_analysis import (
    GATE_NAMES, DEFAULT_CLASH_DISTANCE_GRID, DEFAULT_CLASH_EVENT_GRID,
    alignment_inventory, asset_provenance_manifest, gate_ledger,
    refinement_inventory, ranking_comparison, usalign_preflight,
    leave_one_gate_out,
    validate_usalign_contract, candidate_key, sha256_file,
)
from src.transformation_config import TransformationThresholds

print('Interpreter:', PYTHON)
print('Repository:', REPO_ROOT)


Interpreter: /home/rshadi25/.conda/envs/gtalign_env/bin/python
Repository: /scratch/rshadi25/GitHub/PRISM-prescript


In [2]:
# Edit only this cell to define a comparison. Source assets are not modified.
INPUT_CSV = REPO_ROOT / 'inputs.csv'
# 1gte replay example: INPUT_CSV = REPO_ROOT / 'tmp/agent/20260823-1gte-variant-runs/current/inputs.csv'
TEMPLATE_IDS = ['1zx4AB', '1ngmEF']
# 1gte replay example: TEMPLATE_IDS = ['1h7xCD']
TEMPLATE_LIST = None  # Optional Path to a plain-text template manifest.
ALIGNER = 'tmalign'
SURFACE_BACKEND = 'naccess'
FILTER_MODE = 'geometry_only_experimental'
FILTER_ASSET_ROOT = None  # Set only for a published-protocol replay.
REFINER = 'external_rosetta'
THRESHOLD_OVERRIDES = {}
OUTPUT_PARENT = REPO_ROOT / 'tmp/agent/pipeline-orientation-comparison'
RUN_PIPELINES = False
RUN_CLASH_DIAGNOSTICS = False
EXISTING_RUN_ROOTS = {}  # Optional {'native_default': Path(...), ...}
ARMS = {'native_default': None, 'o1': 'o1', 'o2': 'o2'}

if TEMPLATE_LIST is not None:
    SELECTED_TEMPLATES = select_templates([], template_list_path=TEMPLATE_LIST)
else:
    SELECTED_TEMPLATES = select_templates([], explicit_templates=TEMPLATE_IDS)
BASELINE_THRESHOLDS = TransformationThresholds()
RESOLVED_THRESHOLDS = BASELINE_THRESHOLDS.with_overrides(
    THRESHOLD_OVERRIDES
)

assert INPUT_CSV.exists(), INPUT_CSV
assert SELECTED_TEMPLATES
print('Input CSV:', INPUT_CSV)
print('Templates:', SELECTED_TEMPLATES)
display(pd.DataFrame([{
    'input_csv': str(INPUT_CSV),
    'templates': ','.join(SELECTED_TEMPLATES),
    'aligner': ALIGNER,
    'surface_backend': SURFACE_BACKEND,
    'filter_mode': FILTER_MODE,
    'baseline_thresholds': json.dumps(BASELINE_THRESHOLDS.as_dict(), sort_keys=True),
    'threshold_overrides': json.dumps(THRESHOLD_OVERRIDES, sort_keys=True),
    'run_pipelines': RUN_PIPELINES,
}]))


Input CSV: /scratch/rshadi25/GitHub/PRISM-prescript/inputs.csv
Templates: ['1zx4AB', '1ngmEF']


,input_csv,templates,aligner,surface_backend,filter_mode,baseline_thresholds,threshold_overrides,run_pipelines
0,/scratch/rshadi25/GitHub/PRISM-prescript/input...,"1zx4AB,1ngmEF",tmalign,naccess,geometry_only_experimental,"{""clashing_distance"": 3.0, ""contact_count_thre...",{},False


In [3]:
def build_command(arm_name: str, arm_root: Path):
    command = [
        str(PYTHON), str(CURRENT_CLI),
        '--inputs-csv', str(INPUT_CSV),
        '--templates', *SELECTED_TEMPLATES,
        '--aligner', ALIGNER,
        '--surface-backend', SURFACE_BACKEND,
        '--no-refine',
        '--candidate-audit-path', str(arm_root / 'status/candidates.jsonl'),
    ]
    orientation = ARMS[arm_name]
    if orientation is not None:
        command.extend(['--orientation', orientation])
    for field, value in RESOLVED_THRESHOLDS.as_dict().items():
        command.extend(['--' + field.replace('_', '-'), str(value)])
    return command

PREVIEW_ROOT = OUTPUT_PARENT / 'preview'
display(pd.DataFrame([
    {
        'arm': arm,
        'orientation_option': ARMS[arm] or '(omitted: native default)',
        'command': shlex.join(build_command(arm, PREVIEW_ROOT / arm)),
    }
    for arm in ARMS
]))


,arm,orientation_option,command
0,native_default,(omitted: native default),/home/rshadi25/.conda/envs/gtalign_env/bin/pyt...
1,o1,o1,/home/rshadi25/.conda/envs/gtalign_env/bin/pyt...
2,o2,o2,/home/rshadi25/.conda/envs/gtalign_env/bin/pyt...


In [4]:
def _prepare_arm_root(arm_root: Path):
    arm_root.mkdir(parents=True, exist_ok=True)
    for name in ('templates', 'external_tools'):
        link = arm_root / name
        if not link.exists() and not link.is_symlink():
            link.symlink_to(REPO_ROOT / name, target_is_directory=True)


def _count_transformation_pairs(arm_root: Path):
    directory = arm_root / 'processed/transformation'
    left = {path.name[:-6] for path in directory.glob('*_L.pdb')}
    right = {path.name[:-6] for path in directory.glob('*_R.pdb')}
    return len(left & right)


def run_arm(arm_name: str, comparison_root: Path):
    arm_root = comparison_root / arm_name
    _prepare_arm_root(arm_root)
    status_path = arm_root / 'status/stages.jsonl'
    status_path.parent.mkdir(parents=True, exist_ok=True)
    environment = os.environ.copy()
    environment.pop('PRISM_ORIENTATION', None)
    environment.pop('PRISM_FILTER_ASSET_ROOT', None)
    environment['PRISM_FILTER_MODE'] = FILTER_MODE
    if FILTER_ASSET_ROOT is not None:
        environment['PRISM_FILTER_ASSET_ROOT'] = str(FILTER_ASSET_ROOT)
    environment['PRISM_STAGE_STATUS_PATH'] = str(status_path)
    environment['PRISM_RUN_ID'] = f'{arm_name}-{time.time_ns()}'
    command = build_command(arm_name, arm_root)
    (arm_root / 'status/run_config.json').write_text(json.dumps({
        'arm': arm_name, 'orientation': ARMS[arm_name] or 'native',
        'input_csv': str(INPUT_CSV), 'templates': SELECTED_TEMPLATES,
        'aligner': ALIGNER, 'surface_backend': SURFACE_BACKEND,
        'filter_mode': FILTER_MODE,
        'thresholds': RESOLVED_THRESHOLDS.as_dict(),
        'command': command,
    }, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    started = time.time()
    completed = subprocess.run(
        command, cwd=arm_root, env=environment,
        text=True, capture_output=True, check=False,
    )
    (arm_root / 'status/stdout.log').write_text(completed.stdout, encoding='utf-8')
    (arm_root / 'status/stderr.log').write_text(completed.stderr, encoding='utf-8')
    events = []
    if status_path.exists():
        events = [json.loads(line) for line in status_path.read_text().splitlines() if line.strip()]
    return {
        'arm': arm_name,
        'orientation': ARMS[arm_name] or 'native',
        'return_code': completed.returncode,
        'elapsed_seconds': round(time.time() - started, 2),
        'completed_stages': ','.join(
            event['stage'] for event in events if event.get('event') == 'completed'
        ),
        'transformation_pairs': _count_transformation_pairs(arm_root),
        'run_root': str(arm_root),
        'command': shlex.join(command),
        'thresholds': RESOLVED_THRESHOLDS.as_dict(),
    }


## Execute the three arms (guarded)

The three cells below are intentionally separate so a failed arm is visible and can be inspected before continuing. They share the same `comparison_root`, input CSV, template IDs, aligner, and threshold overrides.

In [5]:
if RUN_PIPELINES:
    ACTIVE_COMPARISON_ROOT = OUTPUT_PARENT / f"{time.strftime('%Y%m%d-%H%M%S')}-{time.time_ns()}"
    ACTIVE_COMPARISON_ROOT.mkdir(parents=True, exist_ok=True)
    native_result = run_arm('native_default', ACTIVE_COMPARISON_ROOT)
    display(pd.DataFrame([native_result]))
else:
    ACTIVE_COMPARISON_ROOT = None
    native_result = None
    print('Native arm disabled; set RUN_PIPELINES = True to execute.')


Native arm disabled; set RUN_PIPELINES = True to execute.


In [6]:
if RUN_PIPELINES:
    o1_result = run_arm('o1', ACTIVE_COMPARISON_ROOT)
    display(pd.DataFrame([o1_result]))
else:
    o1_result = None
    print('o1 arm disabled; set RUN_PIPELINES = True to execute.')


o1 arm disabled; set RUN_PIPELINES = True to execute.


In [7]:
if RUN_PIPELINES:
    o2_result = run_arm('o2', ACTIVE_COMPARISON_ROOT)
    display(pd.DataFrame([o2_result]))
else:
    o2_result = None
    print('o2 arm disabled; set RUN_PIPELINES = True to execute.')


o2 arm disabled; set RUN_PIPELINES = True to execute.


In [8]:
if RUN_PIPELINES:
    comparison_results = pd.DataFrame([native_result, o1_result, o2_result])
    display(comparison_results)
    print('Inspect each run_root/status/stdout.log, stderr.log, stages.jsonl, and candidates.jsonl before interpretation.')
else:
    comparison_results = pd.DataFrame()


## Threshold matrix planning

Use one-variable changes only. A yield change localizes sensitivity; it does not establish improved biological accuracy. Keep benchmark/native labels and all failed-stage statuses attached to the comparison.

In [9]:
THRESHOLD_SWEEP = {
    'current_defaults': {},
    'lower_match_count': {'minimum_residue_match_count': 12},
    'lower_tm_score': {'tm_score_threshold': 0.4},
    'relaxed_clash_count': {'max_clashing_count': 8},
}

display(pd.DataFrame([
    {
        'sweep': name,
        **BASELINE_THRESHOLDS.with_overrides(overrides).as_dict(),
    }
    for name, overrides in THRESHOLD_SWEEP.items()
]))


,sweep,minimum_residue_match_count,minimum_residue_match_percentage,minimum_hotspot_match_number,diff_percentage,template_residue_count,contact_count_threshold,clashing_distance,max_clashing_count,scaffold_threshold,tm_score_threshold,multiprot_minimum_residue_match_count,multiprot_minimum_residue_match_percentage
0,current_defaults,15,50.0,1,20.0,50.0,5,3.0,5,5.0,0.5,10,30.0
1,lower_match_count,12,50.0,1,20.0,50.0,5,3.0,5,5.0,0.5,10,30.0
2,lower_tm_score,15,50.0,1,20.0,50.0,5,3.0,5,5.0,0.4,10,30.0
3,relaxed_clash_count,15,50.0,1,20.0,50.0,5,3.0,8,5.0,0.5,10,30.0


## Stepwise reliability protocol

The cells below implement the recommended diagnostic order. They preserve the production defaults, record missing evidence explicitly, and keep alignment, clash, refinement, ranking, and US-align claims separate. A table is not interpreted as a scientific result until its asset and provenance status is complete.

In [10]:
# Step 1: freeze the current production baseline.
PRODUCTION_BASELINE = BASELINE_THRESHOLDS.as_dict()
CLASH_DISTANCE_GRID = (2.5, 3.0, 3.5)
CLASH_EVENT_GRID = (0, 1, 5, 10, 25, 50, 100)
GATE_NAMES = tuple(GATE_NAMES)
assert PRODUCTION_BASELINE['tm_score_threshold'] == 0.5
assert PRODUCTION_BASELINE['minimum_residue_match_count'] == 15
assert PRODUCTION_BASELINE['minimum_residue_match_percentage'] == 50.0
assert PRODUCTION_BASELINE['diff_percentage'] == 20.0
assert PRODUCTION_BASELINE['clashing_distance'] == 3.0
assert PRODUCTION_BASELINE['max_clashing_count'] == 5
assert PRODUCTION_BASELINE['scaffold_threshold'] == 5.0
display(pd.DataFrame([PRODUCTION_BASELINE]))
print('Diagnostic threshold changes remain opt-in and are never promoted by this notebook.')


,minimum_residue_match_count,minimum_residue_match_percentage,minimum_hotspot_match_number,diff_percentage,template_residue_count,contact_count_threshold,clashing_distance,max_clashing_count,scaffold_threshold,tm_score_threshold,multiprot_minimum_residue_match_count,multiprot_minimum_residue_match_percentage
0,15,50.0,1,20.0,50.0,5,3.0,5,5.0,0.5,10,30.0


Diagnostic threshold changes remain opt-in and are never promoted by this notebook.


## Steps 2–3: inputs, assets, alignment, and pairing

Each arm gets a manifest containing input/source/interface/surface/alignment hashes, backend and filter mode, asset-mixing status, orientation assignments, partner availability, aligner score contracts, return codes, match counts, coverage, native MultiProt RMSD fields, and raw-output hashes. Missing hashes or partners remain visible.

In [11]:
ARM_RESULTS = {
    name: result for name, result in (
        ('native_default', native_result), ('o1', o1_result), ('o2', o2_result)
    ) if result is not None
}
ARM_ROOTS = {name: Path(result['run_root']) for name, result in ARM_RESULTS.items()}
ARM_ROOTS.update({name: Path(path) for name, path in EXISTING_RUN_ROOTS.items()})
ALIGNMENT_ROWS_BY_ARM = {}
PROVENANCE_BY_ARM = {}
for arm, root in ARM_ROOTS.items():
    orientations = ('o1', 'o2') if arm == 'native_default' else (ARMS[arm],)
    audit_path = root / 'status/candidates.jsonl'
    rows = alignment_inventory(
        root, INPUT_CSV, SELECTED_TEMPLATES, arm=arm,
        orientations=orientations, audit_path=audit_path,
    )
    manifest = asset_provenance_manifest(
        root, INPUT_CSV, SELECTED_TEMPLATES, source_root=REPO_ROOT,
        surface_backend=SURFACE_BACKEND, filter_mode=FILTER_MODE,
        filter_asset_root=FILTER_ASSET_ROOT,
    )
    manifest['orientation_partner_availability'] = [
        {key: row[key] for key in ('query_left', 'query_right', 'template', 'orientation', 'partner_left_available', 'partner_right_available', 'partner_available', 'alignment_contract_status_left', 'alignment_contract_status_right', 'alignment_evidence_complete')}
        for row in rows
    ]
    ALIGNMENT_ROWS_BY_ARM[arm] = rows
    PROVENANCE_BY_ARM[arm] = manifest
    status_dir = root / 'status'
    status_dir.mkdir(parents=True, exist_ok=True)
    (status_dir / 'run_manifest.json').write_text(
        json.dumps({
            'arm': arm, 'orientation': 'native' if arm == 'native_default' else arm,
            'input_csv': str(INPUT_CSV), 'templates': SELECTED_TEMPLATES,
            'aligner': ALIGNER, 'surface_backend': SURFACE_BACKEND,
            'filter_mode': FILTER_MODE, 'thresholds': RESOLVED_THRESHOLDS.as_dict(),
            'provenance': manifest,
        }, indent=2, sort_keys=True) + '\n', encoding='utf-8'
    )
    (status_dir / 'alignment_inventory.jsonl').write_text(
        ''.join(json.dumps(row, sort_keys=True) + '\n' for row in rows), encoding='utf-8'
    )
if ARM_ROOTS:
    display(pd.DataFrame([
        {'arm': arm, 'asset_mix_status': manifest['asset_mix_status'],
         'surface_backend': manifest['surface_backend'],
         'surface_outputs': len(manifest['surface_outputs']),
         'alignment_outputs': len(manifest['alignment_outputs'])}
        for arm, manifest in PROVENANCE_BY_ARM.items()
    ]))
else:
    print('No completed or existing run roots supplied; provenance cells are ready but idle.')


No completed or existing run roots supplied; provenance cells are ready but idle.


In [12]:
ALIGNMENT_COLUMNS = [
    'arm', 'orientation', 'query_left', 'query_right', 'template',
    'chain_left', 'chain_right', 'partner_available',
    'alignment_contract_status_left', 'alignment_contract_status_right', 'alignment_evidence_complete',
    'aligner_left', 'score_gate_contract_left', 'return_code_left',
    'match_count_left', 'coverage_left', 'tm_score_left',
    'native_multiprot_score_left', 'raw_output_sha256_left',
    'aligner_right', 'score_gate_contract_right', 'return_code_right',
    'match_count_right', 'coverage_right', 'tm_score_right',
    'native_multiprot_score_right', 'raw_output_sha256_right',
    'raw_output_hash_complete', 'audit_status', 'audit_error_reason',
]
ALIGNMENT_TABLE = pd.DataFrame([
    row for rows in ALIGNMENT_ROWS_BY_ARM.values() for row in rows
])
if not ALIGNMENT_TABLE.empty:
    display(ALIGNMENT_TABLE[ALIGNMENT_COLUMNS])
else:
    print('Alignment ledger is empty until a run root is supplied.')


Alignment ledger is empty until a run root is supplied.


## Step 4: independent gate ablation

The ledger evaluates availability, score contract, match count, coverage, hotspots, complementary contacts, transform materialization, and clash filtering independently. `None` means the gate could not be evaluated or was not applied. The leave-one-gate-out table is descriptive rescue evidence, not permission to relax production.

In [13]:
GATE_ROWS_BY_ARM = {}
for arm, rows in ALIGNMENT_ROWS_BY_ARM.items():
    root = ARM_ROOTS[arm]
    GATE_ROWS_BY_ARM[arm] = gate_ledger(
        root, rows, thresholds=RESOLVED_THRESHOLDS.as_dict(),
        filter_mode=FILTER_MODE, filter_asset_root=FILTER_ASSET_ROOT,
        audit_path=root / 'status/candidates.jsonl',
        compute_clash_diagnostics=RUN_CLASH_DIAGNOSTICS,
    )
    (root / 'status/gate_ledger.jsonl').write_text(
        ''.join(json.dumps(row, sort_keys=True, default=str) + '\n' for row in GATE_ROWS_BY_ARM[arm]),
        encoding='utf-8',
    )
GATE_LEDGER = pd.DataFrame([row for rows in GATE_ROWS_BY_ARM.values() for row in rows])
GATE_DISPLAY = ['arm', 'orientation', 'template', 'query_left', 'query_right', 'audit_status'] + [f'gate_{name}' for name in GATE_NAMES] + ['independent_failures', 'independent_unknown_gates', 'cumulative_pass']
if not GATE_LEDGER.empty:
    display(GATE_LEDGER[GATE_DISPLAY])
    ABLATION_TABLE = pd.DataFrame(leave_one_gate_out(GATE_LEDGER.to_dict('records')))
    display(ABLATION_TABLE.groupby(['arm', 'omitted_gate'], dropna=False)['rescued_if_omitted'].sum().reset_index())
else:
    ABLATION_TABLE = pd.DataFrame()
    print('Gate ledger is empty until a run root is supplied.')


Gate ledger is empty until a run root is supplied.


## Step 5: clash diagnosis before any relaxation

When enabled, the gate ledger calculates event counts, minimum cross-partner distance, residue locations, clashing-residue fractions, a localization heuristic, and the requested distance/event grid. All-atom van der Waals overlap is marked unavailable unless a radii-aware evaluator is supplied.

In [14]:
CLASH_TABLE = pd.DataFrame()
CLASH_GRID_TABLE = pd.DataFrame()
if RUN_CLASH_DIAGNOSTICS and not GATE_LEDGER.empty:
    diagnostics = []
    grids = []
    for row in GATE_LEDGER.to_dict('records'):
        details = row.get('clash_diagnostics')
        if not details:
            continue
        diagnostics.append({
            'arm': row['arm'], 'orientation': row['orientation'],
            'template': row['template'], 'query_left': row['query_left'],
            'query_right': row['query_right'], **{key: value for key, value in details.items() if key != 'grid' and key != 'event_residues'}
        })
        for grid_row in details['grid']:
            grids.append({
                'arm': row['arm'], 'orientation': row['orientation'],
                'template': row['template'], **grid_row
            })
    CLASH_TABLE = pd.DataFrame(diagnostics)
    CLASH_GRID_TABLE = pd.DataFrame(grids)
    for arm, root in ARM_ROOTS.items():
        arm_rows = [row for row in diagnostics if row['arm'] == arm]
        (root / 'status/clash_diagnostics.jsonl').write_text(
            ''.join(json.dumps(row, sort_keys=True, default=str) + '\n' for row in arm_rows), encoding='utf-8'
        )
    display(CLASH_TABLE[['arm', 'orientation', 'template', 'clash_count', 'minimum_cross_partner_distance', 'localization_heuristic', 'all_atom_vdw_status']])
    display(CLASH_GRID_TABLE)
else:
    print('Clash diagnostics deferred; set RUN_CLASH_DIAGNOSTICS = True after reviewing the fixed-panel commands.')


Clash diagnostics deferred; set RUN_CLASH_DIAGNOSTICS = True after reviewing the fixed-panel commands.


## Step 6: independent refinement validation

This validator reports stage and per-candidate return-code availability, output existence, observed chain identities, failure reason, and native-evaluator fields. It never treats the number of output files as a quality result. The comparison arms intentionally use `--no-refine`, so an empty refinement ledger is expected; interpret refinement only after a separately supplied refinement-enabled run. External Rosetta currently lacks per-candidate return-code records, so that limitation remains explicit.

In [15]:
REFINEMENT_ROWS_BY_ARM = {}
for arm, rows in GATE_ROWS_BY_ARM.items():
    REFINEMENT_ROWS_BY_ARM[arm] = refinement_inventory(ARM_ROOTS[arm], rows)
    (ARM_ROOTS[arm] / 'status/refinement_inventory.jsonl').write_text(
        ''.join(json.dumps(row, sort_keys=True, default=str) + '\n' for row in REFINEMENT_ROWS_BY_ARM[arm]), encoding='utf-8'
    )
REFINEMENT_TABLE = pd.DataFrame([row for rows in REFINEMENT_ROWS_BY_ARM.values() for row in rows])
if not REFINEMENT_TABLE.empty:
    display(REFINEMENT_TABLE)
else:
    print('Refinement validation is empty until transformed candidates exist.')


Refinement validation is empty until transformed candidates exist.


## Step 7: ranking only after candidate trust

Populate `RANKING_CANDIDATE_SETS` only with methods run from the exact same pre-ranking candidate set (`none`, `baseline`, `prodigy`). Supply `NATIVE_LABELS` only from an independent native reference and record its source plus SHA-256. Without that provenance, recovery metrics are not treated as validated. PRODIGY affinity is recorded as a ranking input, never as a DockQ/native-like label.

In [16]:
RANKING_CANDIDATE_SETS = {}  # e.g. {'none': [...], 'baseline': [...], 'prodigy': [...]}
RANKING_PRE_RANKING_SETS = {}  # optional method -> exact pre-ranking key list
NATIVE_LABELS = {}  # independent candidate_key -> bool labels; affinity is not accepted here.
NATIVE_LABEL_SOURCE = None  # independent native-label table/manifest path
NATIVE_LABEL_SOURCE_SHA256 = sha256_file(NATIVE_LABEL_SOURCE) if NATIVE_LABEL_SOURCE else None
if RANKING_CANDIDATE_SETS:
    RANKING_TABLE = pd.DataFrame(ranking_comparison(RANKING_CANDIDATE_SETS, pre_ranking_sets=RANKING_PRE_RANKING_SETS or None, native_labels=NATIVE_LABELS, native_label_source=str(NATIVE_LABEL_SOURCE) if NATIVE_LABEL_SOURCE else None, native_label_source_sha256=NATIVE_LABEL_SOURCE_SHA256, top_k=1))
    display(RANKING_TABLE)
else:
    RANKING_TABLE = pd.DataFrame()
    print('Ranking comparison deferred until the no-ranking candidate set is trustworthy and identical across methods.')


Ranking comparison deferred until the no-ranking candidate set is trustworthy and identical across methods.


## Step 8: US-align as a separate validation arm

US-align is not silently inserted into the current PRISM run. First probe availability/version, then run a known pair, validate PRISM-native fields, and only then build a matched downstream comparison.

In [ ]:
USALIGN_EXECUTABLE = 'USalign'
RUN_USALIGN_PREFLIGHT = False
USALIGN_PREFLIGHT = usalign_preflight(USALIGN_EXECUTABLE, probe=RUN_USALIGN_PREFLIGHT)
display(pd.DataFrame([USALIGN_PREFLIGHT]))
if USALIGN_PREFLIGHT['available']:
    print('Availability is not integration: a known-pair JSON probe and PRISM contract adapter are still required.')
else:
    print('US-align executable not found; no US-align pipeline claim is made.')
# Example after a probe: validate_usalign_contract(json.loads(Path('usalign_probe.json').read_text()))


In [ ]:
REQUIREMENT_STATUS = pd.DataFrame([
    {'step': 1, 'requirement': 'frozen production baseline', 'status': 'implemented', 'evidence': 'PRODUCTION_BASELINE'},
    {'step': 2, 'requirement': 'input and asset parity', 'status': 'ready_run_dependent', 'evidence': 'run_manifest.json'},
    {'step': 3, 'requirement': 'alignment and pairing provenance', 'status': 'ready_run_dependent', 'evidence': 'alignment_inventory.jsonl'},
    {'step': 4, 'requirement': 'independent gate ablation', 'status': 'ready_run_dependent', 'evidence': 'gate_ledger.jsonl + ABLATION_TABLE'},
    {'step': 5, 'requirement': 'clash grid and localization', 'status': 'opt_in', 'evidence': 'CLASH_TABLE + CLASH_GRID_TABLE'},
    {'step': 6, 'requirement': 'per-candidate refinement validation', 'status': 'ready_run_dependent', 'evidence': 'refinement_inventory.jsonl'},
    {'step': 7, 'requirement': 'same-set ranking comparison', 'status': 'opt_in_with_native_labels_and_source_hash', 'evidence': 'RANKING_TABLE'},
    {'step': 8, 'requirement': 'separate US-align validation arm', 'status': 'preflight_only', 'evidence': 'USALIGN_PREFLIGHT'},
])
display(REQUIREMENT_STATUS)
print('Interpretation rule: unknown, missing, mixed, deferred, or preflight-only evidence is not a biological success.')
